In [ ]:
import os
import re
import tarfile
import gzip
import json
import random

import ir_datasets
import pandas as pd
import requests
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

GLOBAL_SEED = 1337

DATASET_SPECS = [
    {"year": 2021, "name": "msmarco-passage-v2/trec-dl-2021/judged", "qrels_mode": "ir_datasets"},
    {"year": 2022, "name": "msmarco-passage-v2/trec-dl-2022/judged", "qrels_mode": "ir_datasets"},
    {"year": 2023, "name": "msmarco-passage-v2/trec-dl-2023", "qrels_mode": "trec_file"},
]

TREC_2023_QRELS_URL = "https://trec.nist.gov/data/deep/2023.qrels.pass.withDupes.txt"
TAR_PATH = "msmarco_v2_passage.tar" # assumed downloaded and placed to the path

TARGET_REL = 2
EXPECTED_POOL_SIZE = 100
FINAL_REQUIRED_HIGHLY_RELEVANT = 5

PASSAGE_MIN_CHARS = 40
PASSAGE_VIEW_CHARS = 800
QUERY_VIEW_CHARS = 400

N_TARGET = None
N_PRINT = 5

STOPWORDS = set(ENGLISH_STOP_WORDS)

def norm_ws(s):
    return re.sub(r"\s+", " ", (s or "")).strip()

def load_trec_2023_qrels(url, year):
    r = requests.get(url, timeout=240)
    r.raise_for_status()

    rows = []
    for line in r.text.splitlines():
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 4:
            continue

        query_id = str(parts[0])
        doc_id = str(parts[2])
        relevance = int(parts[3])

        rows.append((query_id, doc_id, relevance, int(year)))

    return pd.DataFrame(rows, columns=["query_id", "doc_id", "relevance", "trec_year"])

def build_prep_rows(work_df, eligible_pairs, query_view_chars, passage_view_chars, global_seed, n_target):
    prep_rows_local = []

    for i, (year, query_id) in enumerate(sorted(eligible_pairs), 1):
        sub = work_df[
            (work_df["trec_year"] == year) &
            (work_df["query_id"] == query_id)
        ].copy()

        sub = sub.sort_values("score", ascending=False).reset_index(drop=True)

        if sub["doc_id"].nunique() != EXPECTED_POOL_SIZE:
            continue

        query_text = norm_ws(sub["query_text"].iloc[0])[:query_view_chars]

        candidates_100 = []
        labels_100 = []

        for _, row in sub.iterrows():
            candidates_100.append({
                "doc_id": str(row["doc_id"]),
                "score": float(row["score"]),
                "passage_text": norm_ws(row["passage_text"])[:passage_view_chars],
                "source_doc_id": row.get("source_doc_id", ""),
            })
            labels_100.append(int(row["relevance"]))

        highly_rel_positions = [ix for ix, rel in enumerate(labels_100) if rel >= TARGET_REL]

        prep_rows_local.append({
            "trec_year": int(year),
            "query_id": str(query_id),
            "query": query_text,
            "candidates_100": candidates_100,
            "labels_100": labels_100,
            "highly_rel_positions": highly_rel_positions,
            "n_candidates": len(candidates_100),
            "n_highly_relevant": len(highly_rel_positions),
        })

        if i % 25 == 0 or i == len(eligible_pairs):
            print(f"Prepared {i}/{len(eligible_pairs)} eligible queries")

    prep_rows_local = sorted(prep_rows_local, key=lambda x: (x["trec_year"], x["query_id"]))

    if n_target is not None:
        rng = random.Random(global_seed)
        rng.shuffle(prep_rows_local)
        prep_rows_local = prep_rows_local[:min(n_target, len(prep_rows_local))]
        prep_rows_local = sorted(prep_rows_local, key=lambda x: (x["trec_year"], x["query_id"]))

    return prep_rows_local

all_queries = []
all_qrels = []
all_scoreddocs = []

print("Loading pooled TREC-DL sets")
print()

for spec in DATASET_SPECS:
    year = spec["year"]
    ds_name = spec["name"]
    qrels_mode = spec["qrels_mode"]

    print(f"Loading year {year}: {ds_name}")
    ds = ir_datasets.load(ds_name)

    queries_year = pd.DataFrame(
        [(str(x.query_id), x.text, int(year)) for x in ds.queries_iter()],
        columns=["query_id", "query_text", "trec_year"]
    )

    scoreddocs_year = pd.DataFrame(
        [(str(x.query_id), str(x.doc_id), float(x.score), int(year)) for x in ds.scoreddocs_iter()],
        columns=["query_id", "doc_id", "score", "trec_year"]
    )

    if qrels_mode == "ir_datasets":
        qrels_year = pd.DataFrame(
            [(str(x.query_id), str(x.doc_id), int(x.relevance), int(year)) for x in ds.qrels_iter()],
            columns=["query_id", "doc_id", "relevance", "trec_year"]
        )
    else:
        qrels_year = load_trec_2023_qrels(TREC_2023_QRELS_URL, year)

    print(f"  queries: {queries_year['query_id'].nunique()}")
    print(f"  qrels rows: {len(qrels_year)}")
    print(f"  scoreddocs rows: {len(scoreddocs_year)}")
    print(f"  unique candidate doc ids: {scoreddocs_year['doc_id'].nunique()}")
    print()

    all_queries.append(queries_year)
    all_qrels.append(qrels_year)
    all_scoreddocs.append(scoreddocs_year)

queries = pd.concat(all_queries, ignore_index=True)
qrels = pd.concat(all_qrels, ignore_index=True)
scoreddocs = pd.concat(all_scoreddocs, ignore_index=True)

queries["query_id"] = queries["query_id"].astype(str)
queries["query_text"] = queries["query_text"].fillna("").astype(str).map(norm_ws)

qrels["query_id"] = qrels["query_id"].astype(str)
qrels["doc_id"] = qrels["doc_id"].astype(str)
qrels["relevance"] = qrels["relevance"].astype(int)

scoreddocs["query_id"] = scoreddocs["query_id"].astype(str)
scoreddocs["doc_id"] = scoreddocs["doc_id"].astype(str)
scoreddocs["score"] = scoreddocs["score"].astype(float)

print("Combined pooled data summary")
print("  total pooled queries:", queries[["trec_year", "query_id"]].drop_duplicates().shape[0])
print("  total pooled qrels rows:", len(qrels))
print("  total pooled scoreddocs rows:", len(scoreddocs))
print("  total pooled unique candidate doc ids:", scoreddocs["doc_id"].nunique())
print()

print("Raw query counts by year")
print(queries.groupby("trec_year")["query_id"].nunique().reset_index(name="raw_queries").to_string(index=False))
print()

selected_doc_ids = set(scoreddocs["doc_id"].unique())

print("Scanning MS MARCO passage tar")
print("  target unique doc ids:", len(selected_doc_ids))
print()

selected_passages = {}

with tarfile.open(TAR_PATH, "r") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]
    print("Inner files:", len(members))
    print()

    for i, member in enumerate(members, 1):
        print(f"Processing tar member {i}/{len(members)}: {member.name}")

        extracted = tar.extractfile(member)
        if extracted is None:
            continue

        if member.name.endswith(".gz"):
            stream = gzip.open(extracted, "rt", encoding="utf-8")
        else:
            stream = extracted

        found_before = len(selected_passages)

        for line in stream:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)
            passage_id = str(row.get("pid"))

            if passage_id in selected_doc_ids:
                selected_passages[passage_id] = row

        stream.close()

        found_after = len(selected_passages)
        print(f"  matched so far: {found_after} | added in this file: {found_after - found_before}")

print()
print("Finished MS MARCO scan")
print("  matched passages found:", len(selected_passages))
print("  coverage ratio:", len(selected_passages) / max(1, len(selected_doc_ids)))
print()

passages = pd.DataFrame([
    {
        "doc_id": str(passage_id),
        "passage_text": row.get("passage", ""),
        "source_doc_id": row.get("docid", ""),
        "spans": row.get("spans", "")
    }
    for passage_id, row in selected_passages.items()
])

passages["doc_id"] = passages["doc_id"].astype(str)
passages["passage_text"] = passages["passage_text"].fillna("").astype(str).map(norm_ws)

print("Passage table summary")
print("  passage rows:", len(passages))
print("  unique doc ids:", passages["doc_id"].nunique())
print()

rerank_df = scoreddocs.merge(
    queries[["query_id", "query_text", "trec_year"]],
    on=["query_id", "trec_year"],
    how="left"
)

rerank_df = rerank_df.merge(
    passages[["doc_id", "passage_text", "source_doc_id", "spans"]],
    on="doc_id",
    how="left"
)

rerank_df = rerank_df.merge(
    qrels[["query_id", "doc_id", "relevance", "trec_year"]],
    on=["query_id", "doc_id", "trec_year"],
    how="left"
)

rerank_df["relevance"] = rerank_df["relevance"].fillna(-1).astype(int)
rerank_df["query_text"] = rerank_df["query_text"].fillna("").astype(str).map(norm_ws)
rerank_df["passage_text"] = rerank_df["passage_text"].fillna("").astype(str).map(norm_ws)
rerank_df["score"] = rerank_df["score"].astype(float)

print("Merged reranking table summary")
print("  total rows:", len(rerank_df))
print("  unique pooled queries:", rerank_df[["trec_year", "query_id"]].drop_duplicates().shape[0])
print("  rows with passage text:", rerank_df["passage_text"].notna().sum())
print("  rows with non-empty passage text:", (rerank_df["passage_text"].str.len() > 0).sum())
print()

work_df = rerank_df.copy()
work_df = work_df[work_df["passage_text"].str.len() >= PASSAGE_MIN_CHARS].copy()

print("After minimum passage length filter")
print("  rows:", len(work_df))
print("  unique pooled queries:", work_df[["trec_year", "query_id"]].drop_duplicates().shape[0])
print()

candidate_pairs = work_df[["trec_year", "query_id", "doc_id"]].drop_duplicates()
qrel_pairs = qrels[["trec_year", "query_id", "doc_id", "relevance"]].drop_duplicates()

cand_with_labels = candidate_pairs.merge(
    qrel_pairs,
    on=["trec_year", "query_id", "doc_id"],
    how="left"
)

cand_with_labels["relevance"] = cand_with_labels["relevance"].fillna(-1).astype(int)

query_pool_stats = (
    cand_with_labels.groupby(["trec_year", "query_id"])
    .agg(
        candidate_count=("doc_id", "nunique"),
        n_rel_ge_2=("relevance", lambda s: int((s >= 2).sum()))
    )
    .reset_index()
)

eligible_k3 = query_pool_stats[
    (query_pool_stats["candidate_count"] == EXPECTED_POOL_SIZE) &
    (query_pool_stats["n_rel_ge_2"] >= 3)
].copy()

eligible_k5 = query_pool_stats[
    (query_pool_stats["candidate_count"] == EXPECTED_POOL_SIZE) &
    (query_pool_stats["n_rel_ge_2"] >= 5)
].copy()

summary_df = (
    query_pool_stats.groupby("trec_year")
    .agg(
        raw_queries=("query_id", "nunique"),
        queries_with_100_candidates=("candidate_count", lambda s: int((s == EXPECTED_POOL_SIZE).sum())),
        min_rel_ge_2=("n_rel_ge_2", "min"),
        median_rel_ge_2=("n_rel_ge_2", "median"),
        mean_rel_ge_2=("n_rel_ge_2", "mean"),
        max_rel_ge_2=("n_rel_ge_2", "max")
    )
    .reset_index()
)

k3_counts = eligible_k3.groupby("trec_year").size().reset_index(name="eligible_k3")
k5_counts = eligible_k5.groupby("trec_year").size().reset_index(name="eligible_k5")

summary_df = summary_df.merge(k3_counts, on="trec_year", how="left")
summary_df = summary_df.merge(k5_counts, on="trec_year", how="left")
summary_df["eligible_k3"] = summary_df["eligible_k3"].fillna(0).astype(int)
summary_df["eligible_k5"] = summary_df["eligible_k5"].fillna(0).astype(int)
summary_df["drop_k3_to_k5"] = summary_df["eligible_k3"] - summary_df["eligible_k5"]
summary_df["keep_rate_k5_from_k3"] = summary_df["eligible_k5"] / summary_df["eligible_k3"].replace(0, pd.NA)

print("Eligibility comparison for pooled TREC-DL years")
print()
print(summary_df.to_string(index=False))
print()

print("Pooled totals")
print("  eligible at K=3:", len(eligible_k3))
print("  eligible at K=5:", len(eligible_k5))
print("  final experiment set will use K=5-eligible queries only")
print()

eligible_pairs_final = list(
    zip(
        eligible_k5["trec_year"].astype(int),
        eligible_k5["query_id"].astype(str)
    )
)

prep_rows = build_prep_rows(
    work_df=work_df,
    eligible_pairs=eligible_pairs_final,
    query_view_chars=QUERY_VIEW_CHARS,
    passage_view_chars=PASSAGE_VIEW_CHARS,
    global_seed=GLOBAL_SEED,
    n_target=N_TARGET,
)

print()
print("Final prepared dataset summary")
print("  prepared queries:", len(prep_rows))
print()

if prep_rows:
    shape_df = pd.DataFrame([
        {
            "trec_year": x["trec_year"],
            "query_id": x["query_id"],
            "n_candidates": x["n_candidates"],
            "n_highly_relevant": x["n_highly_relevant"]
        }
        for x in prep_rows
    ])

    print("Prepared queries by year")
    print(shape_df.groupby("trec_year").size().reset_index(name="prepared_queries").to_string(index=False))
    print()

    print("Prepared query shape summary")
    print(shape_df[["n_candidates", "n_highly_relevant"]].describe().to_string())
    print()

    for ex in prep_rows[:N_PRINT]:
        print("=" * 120)
        print("TREC YEAR:", ex["trec_year"])
        print("QUERY ID:", ex["query_id"])
        print("QUERY:", ex["query"])
        print("N CANDIDATES:", ex["n_candidates"])
        print("N HIGHLY RELEVANT:", ex["n_highly_relevant"])
        print("TOP HIGHLY RELEVANT POSITIONS:", ex["highly_rel_positions"][:10])
        print()

        preview_rows = []
        for j, c in enumerate(ex["candidates_100"][:5]):
            preview_rows.append({
                "cand_ix": j,
                "doc_id": c["doc_id"],
                "score": round(c["score"], 4),
                "relevance": ex["labels_100"][j],
                "passage_preview": c["passage_text"][:220]
            })

        print(pd.DataFrame(preview_rows).to_string(index=False))
        print()

Loading pooled TREC-DL sets

Loading year 2021: msmarco-passage-v2/trec-dl-2021/judged
  queries: 53
  qrels rows: 10828
  scoreddocs rows: 5300
  unique candidate doc ids: 5299

Loading year 2022: msmarco-passage-v2/trec-dl-2022/judged
  queries: 76
  qrels rows: 386416
  scoreddocs rows: 7600
  unique candidate doc ids: 7580

Loading year 2023: msmarco-passage-v2/trec-dl-2023
  queries: 700
  qrels rows: 22327
  scoreddocs rows: 70000
  unique candidate doc ids: 67737

Combined pooled data summary
  total pooled queries: 829
  total pooled qrels rows: 419571
  total pooled scoreddocs rows: 82900
  total pooled unique candidate doc ids: 80414

Raw query counts by year
 trec_year  raw_queries
      2021           53
      2022           76
      2023          700

Scanning MS MARCO passage tar
  target unique doc ids: 80414

Inner files: 70

Processing tar member 1/70: msmarco_v2_passage/msmarco_passage_00.gz
  matched so far: 1319 | added in this file: 1319
Processing tar member 2/70:

In [6]:
import os
import json
from datetime import datetime, UTC
import pandas as pd

EXPORT_ROOT = os.path.join(os.getcwd(), "colab_exports")
os.makedirs(EXPORT_ROOT, exist_ok=True)

EXPORT_NAME = "trecdl_2021_2023_k5eligible_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
EXPORT_DIR = os.path.join(EXPORT_ROOT, EXPORT_NAME)
os.makedirs(EXPORT_DIR, exist_ok=True)

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

export_rows = []

for ex in prep_rows:
    clean_candidates = []

    for c in ex["candidates_100"]:
        clean_candidate = {
            "doc_id": c["doc_id"],
            "score": c["score"],
            "passage_text": c["passage_text"]
        }

        if "source_doc_id" in c:
            clean_candidate["source_doc_id"] = c["source_doc_id"]

        clean_candidates.append(clean_candidate)

    export_rows.append({
        "trec_year": ex["trec_year"],
        "query_id": ex["query_id"],
        "query": ex["query"],
        "labels_100": ex["labels_100"],
        "highly_rel_positions": ex["highly_rel_positions"],
        "n_candidates": ex["n_candidates"],
        "n_highly_relevant": ex["n_highly_relevant"],
        "candidates_100": clean_candidates
    })

dataset_jsonl_path = os.path.join(EXPORT_DIR, "prep_rows.jsonl")
write_jsonl(dataset_jsonl_path, export_rows)

meta = {
    "export_name": EXPORT_NAME,
    "created_utc": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S"),
    "years_included": [2021, 2022, 2023],
    "eligibility_rule": "candidate_count == 100 and n_rel_ge_2 >= 5",
    "num_queries": len(export_rows),
    "fields_per_row": [
        "trec_year",
        "query_id",
        "query",
        "labels_100",
        "highly_rel_positions",
        "n_candidates",
        "n_highly_relevant",
        "candidates_100"
    ],
    "candidate_fields": [
        "doc_id",
        "score",
        "passage_text",
        "source_doc_id"
    ]
}
write_json(os.path.join(EXPORT_DIR, "meta.json"), meta)

preview_rows = []
for row in export_rows[:20]:
    preview_rows.append({
        "trec_year": row["trec_year"],
        "query_id": row["query_id"],
        "query": row["query"],
        "n_candidates": row["n_candidates"],
        "n_highly_relevant": row["n_highly_relevant"],
        "first_passage_preview": row["candidates_100"][0]["passage_text"][:200] if row["candidates_100"] else ""
    })

preview_df = pd.DataFrame(preview_rows)
preview_csv_path = os.path.join(EXPORT_DIR, "preview.csv")
preview_df.to_csv(preview_csv_path, index=False)

print("Export finished")
print("Export directory:", EXPORT_DIR)
print("JSONL dataset:", dataset_jsonl_path)
print("Metadata:", os.path.join(EXPORT_DIR, "meta.json"))
print("Preview CSV:", preview_csv_path)

print()
print("Sample row keys:", list(export_rows[0].keys()) if export_rows else [])
if export_rows and export_rows[0]["candidates_100"]:
    print("Sample candidate keys:", list(export_rows[0]["candidates_100"][0].keys()))

Export finished
Export directory: /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_exports/trecdl_2021_2023_k5eligible_20260412_221209
JSONL dataset: /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_exports/trecdl_2021_2023_k5eligible_20260412_221209/prep_rows.jsonl
Metadata: /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_exports/trecdl_2021_2023_k5eligible_20260412_221209/meta.json
Preview CSV: /home/baris/projects/submitted_multinews_llm_ranking_public-Revised/colab_exports/trecdl_2021_2023_k5eligible_20260412_221209/preview.csv

Sample row keys: ['trec_year', 'query_id', 'query', 'labels_100', 'highly_rel_positions', 'n_candidates', 'n_highly_relevant', 'candidates_100']
Sample candidate keys: ['doc_id', 'score', 'passage_text', 'source_doc_id']
